# 01c – Round-1 patches with the current class scheme

The round-1 patches on the university JupyterHub were cut by an **earlier version** of notebook 01.
They contain the same 33 areas and the same patches (train 2,920 / validation 864 / test 864) as the
round-1 training, but with the old class scheme: WorldCover 70 (snow and ice) was merged into
"Bare / snow". From round 2 on, snow is out of scope and set to 255 (ignored).

This notebook re-cuts the **labels** of every round-1 patch from the raw WorldCover files of that run,
with the current scheme. The radar patches are not changed (they are linked, not copied).

A check makes sure that snow is the **only** difference between old and new labels. If anything else
differs, the raw files do not belong to these patches and the notebook stops.

If `data/patches_v2` already exists, the notebook only checks it again.

In [2]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio==1.4.4"], check=True)
print("rasterio ready")

rasterio ready


In [3]:
import os
import json
from pathlib import Path

import numpy as np
import rasterio

OLD_PATCHES = Path("/home/jovyan/vault/Deep Learning/data/patches")   # round-1 patches, old class scheme
RAW_FOLDER = Path("/home/jovyan/vault/Deep Learning/data/raw")        # raw files of that run
NEW_PATCHES = Path("data/patches_v2")
PATCH_SIZE = 256
IGNORE = 255

CLASSES = ["Trees (incl. mangroves)", "Grassland / shrubland", "Cropland", "Built-up",
           "Bare / sparse vegetation", "Permanent water", "Herbaceous wetland"]
REMAP = {10: 0, 95: 0, 30: 1, 20: 1, 100: 1, 40: 2, 50: 3, 60: 4, 80: 5, 90: 6}   # 70 snow -> IGNORE
OLD_BARE_SNOW = 4

old_info = json.loads((OLD_PATCHES / "classes.json").read_text())
print("old classes:", old_info["classes"])
assert old_info.get("remap", {}).get("70") == OLD_BARE_SNOW, "these are not the old-scheme patches"

old classes: ['Trees', 'Grass / shrub', 'Cropland', 'Built-up', 'Bare / snow', 'Water', 'Wetland']


In [4]:
lookup = np.full(101, IGNORE, dtype=np.uint8)
for code, new_class in REMAP.items():
    lookup[code] = new_class


def to_classes(codes):
    classes = np.full(codes.shape, IGNORE, dtype=np.uint8)
    known = (codes >= 0) & (codes <= 100)
    classes[known] = lookup[codes[known]]
    return classes


labels = {}
def worldcover(area):
    if area not in labels:
        with rasterio.open(RAW_FOLDER / area / "worldcover_2021.tif") as src:
            labels[area] = to_classes(src.read(1))
    return labels[area]

In [5]:
NEW_PATCHES.mkdir(parents=True, exist_ok=True)

for split in ["train", "validation", "test"]:
    old_y = np.load(OLD_PATCHES / f"{split}_y.npy")
    rows = (OLD_PATCHES / f"{split}_meta.csv").read_text().strip().split("\n")[1:]

    new_y = np.empty_like(old_y)
    for i, row in enumerate(rows):
        area, continent, month, r, c = row.split(",")
        r, c = int(r), int(c)
        new_y[i] = worldcover(area)[r:r + PATCH_SIZE, c:c + PATCH_SIZE]

    # the only allowed difference: snow (old "Bare / snow") -> IGNORE
    changed = new_y != old_y
    assert np.all((old_y[changed] == OLD_BARE_SNOW) & (new_y[changed] == IGNORE)), \
        f"{split}: labels differ by more than snow - the raw files do not belong to these patches"

    target_y = NEW_PATCHES / f"{split}_y.npy"
    if target_y.exists():
        assert np.array_equal(np.load(target_y), new_y), f"{target_y} differs from a fresh re-cut"
        state = "checked (already existed)"
    else:
        np.save(target_y, new_y)
        state = "created"

    for name in [f"{split}_x.npy", f"{split}_meta.csv"]:          # radar and meta are unchanged
        target = NEW_PATCHES / name
        if not target.exists():
            os.symlink((OLD_PATCHES / name).resolve(), target)
        assert target.resolve() == (OLD_PATCHES / name).resolve(), f"{target} points to the wrong file"

    print(f"{split:<11} {len(rows):5d} patches, snow pixels set to ignore: {int(changed.sum()):7d}  -> {state}")

train        2920 patches, snow pixels set to ignore:   26084  -> checked (already existed)
validation    864 patches, snow pixels set to ignore:       0  -> checked (already existed)
test          864 patches, snow pixels set to ignore:       0  -> checked (already existed)


In [6]:
(NEW_PATCHES / "classes.json").write_text(json.dumps({
    "classes": CLASSES,
    "worldcover_to_class": {str(code): new for code, new in REMAP.items()},
    "ignore": IGNORE,
    "out_of_scope": {"70": "Snow and ice"},
    "plan_version": "v2",
    "note": "labels re-cut from the round-1 raw WorldCover files with the round-2 class scheme",
}, indent=2))

total = sum(np.load(NEW_PATCHES / f"{s}_y.npy", mmap_mode="r").size for s in ["train", "validation", "test"])
print("files:", sorted(p.name for p in NEW_PATCHES.iterdir()))
print(f"{total:,} label pixels in total")

# 02b must see exactly one round-1 set: no other folder in data/ may contain the old patches
leftovers = [p for p in Path("data").iterdir() if p.is_symlink() and p.resolve() == OLD_PATCHES.resolve()]
for link in leftovers:
    link.unlink()
    print("removed old link:", link)

files: ['classes.json', 'test_meta.csv', 'test_x.npy', 'test_y.npy', 'train_meta.csv', 'train_x.npy', 'train_y.npy', 'validation_meta.csv', 'validation_x.npy', 'validation_y.npy']
304,611,328 label pixels in total
